# Spotify API Year Enrichment

This notebook gets real release years from the Spotify Web API using `track_id` values from the Kaggle dataset. It writes `dataset_5a/spotify_api_track_years.csv`, which the preprocessing notebook can use as the source for the `Year` column.

Before running it, create a Spotify Developer app and set `SPOTIFY_CLIENT_ID` and `SPOTIFY_CLIENT_SECRET` as environment variables.


## 1. Import Libraries


In [ ]:
import base64
import json
import os
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd


## 2. Locate Dataset and Output File


In [ ]:
PROJECT_DIR = Path.cwd()
DASH_DIR = PROJECT_DIR if PROJECT_DIR.name == "spotify_trends_dashboard" else PROJECT_DIR / "spotify_trends_dashboard"
if not DASH_DIR.exists():
    DASH_DIR = PROJECT_DIR

DATA_DIR = DASH_DIR / "dataset_5a"
DATA_DIR.mkdir(exist_ok=True)

candidates = [
    PROJECT_DIR / "dataset.csv",
    PROJECT_DIR.parent / "dataset.csv",
    DASH_DIR / "dataset.csv",
]

raw_path = next((p for p in candidates if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No dataset.csv found in the project root or dashboard folder.")

api_years_path = DATA_DIR / "spotify_api_track_years.csv"

print(f"Using dataset: {raw_path}")
print(f"API years output: {api_years_path}")


## 3. Load Track IDs


In [ ]:
raw_df = pd.read_csv(raw_path, low_memory=False)
raw_df.columns = [c.strip() for c in raw_df.columns]

if "track_id" not in raw_df.columns:
    raise KeyError("dataset.csv must contain a track_id column for Spotify API enrichment.")

track_ids = raw_df["track_id"].dropna().astype(str).str.strip()
track_ids = track_ids[track_ids.ne("")].drop_duplicates().tolist()

print(f"Unique track IDs: {len(track_ids):,}")


## 4. Authenticate with Spotify

This uses the Client Credentials flow. The secret stays outside the notebook in environment variables.


In [ ]:
client_id = os.getenv("SPOTIFY_CLIENT_ID", "").strip()
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET", "").strip()

if not client_id or not client_secret:
    raise RuntimeError("Set SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET before running this notebook.")

auth_value = base64.b64encode(f"{client_id}:{client_secret}".encode("utf-8")).decode("utf-8")
request = Request(
    "https://accounts.spotify.com/api/token",
    data=urlencode({"grant_type": "client_credentials"}).encode("utf-8"),
    headers={
        "Authorization": f"Basic {auth_value}",
        "Content-Type": "application/x-www-form-urlencoded",
    },
    method="POST",
)

with urlopen(request, timeout=30) as response:
    token_payload = json.loads(response.read().decode("utf-8"))

access_token = token_payload["access_token"]
print("Spotify access token received.")


## 5. Fetch Release Years

The Spotify Tracks endpoint accepts up to 50 track IDs per request. The notebook saves progress during the run, so it can resume if interrupted. Set `SPOTIFY_API_MAX_TRACKS` to a number for testing, or leave it unset to process all IDs.


In [ ]:
def extract_year(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if len(text) < 4 or not text[:4].isdigit():
        return None
    year = int(text[:4])
    if 1900 <= year <= 2035:
        return year
    return None


def save_progress(records, output_path):
    output_df = pd.DataFrame(records).drop_duplicates("track_id", keep="last")
    output_df = output_df.sort_values("track_id")
    output_df.to_csv(output_path, index=False)


existing_records = []
if api_years_path.exists():
    existing_df = pd.read_csv(api_years_path)
    if {"track_id", "Year"}.issubset(existing_df.columns):
        existing_df = existing_df[["track_id", "Year"]].dropna().copy()
        existing_df["track_id"] = existing_df["track_id"].astype(str)
        existing_df["Year"] = existing_df["Year"].astype(int)
        existing_records = existing_df.to_dict("records")

records = existing_records.copy()
completed_ids = {record["track_id"] for record in records}
remaining_ids = [track_id for track_id in track_ids if track_id not in completed_ids]

max_tracks = os.getenv("SPOTIFY_API_MAX_TRACKS", "").strip()
if max_tracks:
    remaining_ids = remaining_ids[:int(max_tracks)]

batch_size = 50
for start in range(0, len(remaining_ids), batch_size):
    batch = remaining_ids[start:start + batch_size]
    query = urlencode({"ids": ",".join(batch)})
    request = Request(
        f"https://api.spotify.com/v1/tracks?{query}",
        headers={"Authorization": f"Bearer {access_token}"},
        method="GET",
    )
    try:
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as error:
        if error.code == 429:
            retry_after = int(error.headers.get("Retry-After", "5"))
            time.sleep(retry_after)
            with urlopen(request, timeout=30) as response:
                payload = json.loads(response.read().decode("utf-8"))
        else:
            raise
    except URLError:
        save_progress(records, api_years_path)
        raise

    for item in payload.get("tracks", []):
        if item and item.get("id"):
            year = extract_year(item.get("album", {}).get("release_date"))
            if year is not None:
                records.append({"track_id": item["id"], "Year": year})

    if start % 1000 == 0:
        save_progress(records, api_years_path)
        print(f"Saved {len(records):,} years after {start + len(batch):,} requested IDs.")

    time.sleep(0.15)

save_progress(records, api_years_path)
print(f"Saved final API years file with {len(records):,} rows.")


## 6. Next Step

After this file is created, run `five_a_data_prep.ipynb`. The preprocessing notebook will use `spotify_api_track_years.csv` as the real source for `Year`.
